# FHIR Encounter – Incremental Ingestion

Fetch paginated Encounter resources from the public FHIR R4 API, preserve each original JSON Bundle in the Lakehouse Raw layer, and merge resource versions into the Bronze Delta table.

This notebook uses a persistent watermark, a one-hour overlap window, run-specific raw folders, and ingestion metadata logging. The watermark advances only after the Bronze merge succeeds.

## Cell 1 – Create shared control tables

In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.fhir_watermark (
    resource_type STRING,
    last_successful_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.fhir_ingestion_log (
    run_id STRING,
    resource_name STRING,
    api_url_or_params STRING,
    extraction_timestamp TIMESTAMP,
    load_timestamp TIMESTAMP,
    pages_fetched BIGINT,
    records_fetched BIGINT,
    records_saved BIGINT,
    raw_folder_path STRING,
    execution_status STRING,
    error_message STRING
)
USING DELTA
""")

print("Watermark and ingestion log tables are ready.")

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 3, Finished, Available, Finished, False)

Watermark and ingestion log tables are ready.


## Cell 2 – Imports, configuration and watermark

In [2]:
import requests
import json
import uuid
from datetime import datetime, timezone, timedelta

from pyspark.sql.functions import (
    col, explode, to_json, current_timestamp, lit, input_file_name
)
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)
from delta.tables import DeltaTable

# FHIR API CONFIGURATION
FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCE_TYPE = "Encounter"
PAGE_SIZE = 100
OVERLAP_HOURS = 1

# CREATE A UNIQUE ID FOR THIS RUN
run_id = str(uuid.uuid4())
extraction_timestamp = datetime.now(timezone.utc)
extraction_timestamp_naive = extraction_timestamp.replace(tzinfo=None)

# READ THE LAST SUCCESSFUL WATERMARK
watermark_row = spark.sql(f"""
    SELECT last_successful_timestamp
    FROM bronze.fhir_watermark
    WHERE resource_type = '{RESOURCE_TYPE}'
""").first()

# FIRST RUN: LAST THREE DAYS
# LATER RUNS: ONE-HOUR OVERLAP FROM LAST SUCCESSFUL WATERMARK
if watermark_row is None or watermark_row["last_successful_timestamp"] is None:
    start_date = extraction_timestamp - timedelta(days=3)
else:
    last_successful = watermark_row["last_successful_timestamp"].replace(
        tzinfo=timezone.utc
    )
    start_date = last_successful - timedelta(hours=OVERLAP_HOURS)

start_time = start_date.strftime("%Y-%m-%dT%H:%M:%SZ")
end_time = extraction_timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

api_url = f"{FHIR_BASE_URL}/{RESOURCE_TYPE}"
params = [
    ("_count", str(PAGE_SIZE)),
    ("_lastUpdated", f"ge{start_time}"),
    ("_lastUpdated", f"lt{end_time}"),
    ("_format", "json")
]

headers = {"Accept": "application/fhir+json"}
api_details = f"{api_url} | Parameters: {json.dumps(params)}"

# UNIQUE RUN FOLDER PREVENTS SAME-DAY RUNS FROM OVERWRITING RAW FILES
extraction_date = extraction_timestamp.strftime("%Y-%m-%d")
raw_folder = (
    f"Files/raw/{RESOURCE_TYPE}/"
    f"extraction_date={extraction_date}/"
    f"run_id={run_id}"
)

print("Resource:", RESOURCE_TYPE)
print("Run ID:", run_id)
print("Start date:", start_time)
print("End date:", end_time)
print("Raw folder:", raw_folder)

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 4, Finished, Available, Finished, False)

Resource: Encounter
Run ID: 9f0257f1-c90d-4edf-b625-4f5cf12c59e3
Start date: 2026-11-15T20:02:04Z
End date: 2026-11-18T20:02:04Z
Raw folder: Files/raw/Encounter/extraction_date=2026-11-18/run_id=9f0257f1-c90d-4edf-b625-4f5cf12c59e3


## Cell 3 – Metadata logging helper and start the run

In [3]:
log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("resource_name", StringType(), False),
    StructField("api_url_or_params", StringType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), True),
    StructField("pages_fetched", LongType(), True),
    StructField("records_fetched", LongType(), True),
    StructField("records_saved", LongType(), True),
    StructField("raw_folder_path", StringType(), True),
    StructField("execution_status", StringType(), False),
    StructField("error_message", StringType(), True)
])

def write_ingestion_log(status, pages=0, fetched=0, saved=0, error_message=None):
    log_row = [(
        run_id,
        RESOURCE_TYPE,
        api_details,
        extraction_timestamp_naive,
        datetime.now(timezone.utc).replace(tzinfo=None),
        int(pages),
        int(fetched),
        int(saved),
        raw_folder,
        status,
        error_message
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)
    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze.fhir_ingestion_log")
    )

write_ingestion_log("STARTED")
print("Ingestion run logged.")

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 5, Finished, Available, Finished, False)

Ingestion run logged.


## Cell 4 – Fetch paginated API responses and save original JSON

In [4]:
notebookutils.fs.mkdirs(raw_folder)

next_url = api_url
next_params = params
page_number = 1
total_records = 0

try:
    while next_url:
        print(f"Fetching page {page_number}...")

        response = requests.get(
            next_url,
            headers=headers,
            params=next_params,
            timeout=120
        )
        response.raise_for_status()

        # SAVE THE ORIGINAL RESPONSE TEXT WITHOUT MODIFYING IT
        raw_json = response.text
        bundle = response.json()

        file_path = f"{raw_folder}/page_{page_number:04d}.json"
        notebookutils.fs.put(file_path, raw_json, True)

        records_in_page = len(bundle.get("entry", []))
        total_records += records_in_page
        print(f"Page {page_number}: {records_in_page} records saved")

        next_link = next(
            (
                link["url"]
                for link in bundle.get("link", [])
                if link.get("relation") == "next"
            ),
            None
        )

        # NEXT LINK ALREADY CONTAINS ITS QUERY PARAMETERS
        next_url = next_link
        next_params = None
        page_number += 1

    total_pages = page_number - 1
    print("Pages fetched:", total_pages)
    print("Records fetched:", total_records)
    print("Raw folder:", raw_folder)

except Exception as exc:
    write_ingestion_log(
        "FAILED",
        pages=page_number - 1,
        fetched=total_records,
        error_message=str(exc)[:4000]
    )
    raise

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 6, Finished, Available, Finished, False)

Fetching page 1...
Page 1: 100 records saved
Fetching page 2...
Page 2: 100 records saved
Fetching page 3...
Page 3: 100 records saved
Fetching page 4...
Page 4: 100 records saved
Fetching page 5...
Page 5: 100 records saved
Fetching page 6...
Page 6: 100 records saved
Fetching page 7...
Page 7: 100 records saved
Fetching page 8...
Page 8: 100 records saved
Fetching page 9...
Page 9: 100 records saved
Fetching page 10...
Page 10: 100 records saved
Fetching page 11...
Page 11: 100 records saved
Fetching page 12...
Page 12: 100 records saved
Fetching page 13...
Page 13: 100 records saved
Fetching page 14...
Page 14: 100 records saved
Fetching page 15...
Page 15: 100 records saved
Fetching page 16...
Page 16: 100 records saved
Fetching page 17...
Page 17: 100 records saved
Fetching page 18...
Page 18: 100 records saved
Fetching page 19...
Page 19: 100 records saved
Fetching page 20...
Page 20: 97 records saved
Pages fetched: 20
Records fetched: 1997
Raw folder: Files/raw/Encounter/extract

## Cell 5 – Read the raw JSON Bundles

In [5]:
raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(raw_folder)
)

print("Raw Bundle files loaded.")
raw_df.printSchema()

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 7, Finished, Available, Finished, False)

Raw Bundle files loaded.
root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- appointment: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- reference: string (nullable = true)
 |    |    |    |-- class: struct (nullable = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |-- valueBoolean: boolean (nullable = true)
 |    |    |    |    |    |-- valueString: string (nullable = true)
 |    |    |    |-- hospitalization: struct (nullable = true

## Cell 6 – Extract individual Encounter resources

In [6]:
encounter_df = (
    raw_df
    .select(explode(col("entry")).alias("entry"))
    .select(col("entry.resource").alias("resource"))
)

print("Encounter resources extracted.")
print("Resource records:", encounter_df.count())

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 8, Finished, Available, Finished, False)

Encounter resources extracted.
Resource records: 1997


## Cell 7 – Prepare the consistent seven-column Bronze DataFrame

In [7]:
# KEEP THE COMPLETE FHIR RESOURCE IN resource_json
bronze_encounter_df = (
    encounter_df
    .select(
        col("resource.*"),
        to_json(col("resource")).alias("resource_json")
    )
    .withColumn(
        "extraction_timestamp",
        lit(extraction_timestamp_naive).cast("timestamp")
    )
    .withColumn("api_url_or_params", lit(api_details))
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_file", input_file_name())
)

bronze_encounter_final = (
    bronze_encounter_df
    .select(
        "id",
        "resourceType",
        "resource_json",
        "extraction_timestamp",
        "api_url_or_params",
        "load_timestamp",
        "source_file"
    )
    .dropDuplicates(["id", "resource_json"])
)

print("Bronze DataFrame prepared.")
print("Incoming records:", bronze_encounter_final.count())
bronze_encounter_final.printSchema()

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 9, Finished, Available, Finished, False)

Bronze DataFrame prepared.
Incoming records: 1997
root
 |-- id: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource_json: string (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = false)
 |-- api_url_or_params: string (nullable = false)
 |-- load_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



## Cell 8 – Merge Bronze data, then advance the watermark

In [8]:
try:
    target_table = "bronze.encounter"

    if not spark.catalog.tableExists(target_table):
        before_count = 0
        (
            bronze_encounter_final.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        before_count = spark.table(target_table).count()
        bronze_table = DeltaTable.forName(spark, target_table)

        (
            bronze_table.alias("target")
            .merge(
                bronze_encounter_final.alias("source"),
                """
                target.id = source.id
                AND target.resource_json = source.resource_json
                """
            )
            .whenNotMatchedInsert(
                values={
                    "id": "source.id",
                    "resourceType": "source.resourceType",
                    "resource_json": "source.resource_json",
                    "extraction_timestamp": "source.extraction_timestamp",
                    "api_url_or_params": "source.api_url_or_params",
                    "load_timestamp": "source.load_timestamp",
                    "source_file": "source.source_file"
                }
            )
            .execute()
        )

    after_count = spark.table(target_table).count()
    records_saved = after_count - before_count

    print("Bronze Encounter merge completed.")
    print("New resource versions inserted:", records_saved)
    print("Total Bronze records:", spark.table(target_table).count())

    # ADVANCE WATERMARK ONLY AFTER RAW AND BRONZE DATA ARE SAVED
    watermark_update_df = spark.createDataFrame(
        [(RESOURCE_TYPE, extraction_timestamp_naive)],
        ["resource_type", "last_successful_timestamp"]
    )

    watermark_table = DeltaTable.forName(spark, "bronze.fhir_watermark")

    (
        watermark_table.alias("target")
        .merge(
            watermark_update_df.alias("source"),
            "target.resource_type = source.resource_type"
        )
        .whenMatchedUpdate(
            set={
                "last_successful_timestamp": "source.last_successful_timestamp",
                "updated_at": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "resource_type": "source.resource_type",
                "last_successful_timestamp": "source.last_successful_timestamp",
                "updated_at": "current_timestamp()"
            }
        )
        .execute()
    )

    write_ingestion_log(
        "SUCCESS",
        pages=total_pages,
        fetched=total_records,
        saved=records_saved
    )

    print("Encounter watermark updated successfully.")

except Exception as exc:
    write_ingestion_log(
        "FAILED",
        pages=locals().get("total_pages", 0),
        fetched=locals().get("total_records", 0),
        saved=locals().get("records_saved", 0),
        error_message=str(exc)[:4000]
    )
    raise

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 10, Finished, Available, Finished, False)

Bronze Encounter merge completed.
New resource versions inserted: 2
Total Bronze records: 2005
Encounter watermark updated successfully.


## Cell 9 – Verify Bronze, watermark and ingestion log

In [9]:
print("BRONZE ENCOUNTER VALIDATION")

spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT id) AS unique_ids,
        COUNT(*) - COUNT(id) AS null_ids
    FROM bronze.encounter
""").show()

print("WATERMARK")
spark.sql("""
    SELECT *
    FROM bronze.fhir_watermark
    WHERE resource_type = 'Encounter'
""").show(truncate=False)

print("INGESTION LOG")
spark.sql(f"""
    SELECT *
    FROM bronze.fhir_ingestion_log
    WHERE run_id = '{run_id}'
    ORDER BY load_timestamp DESC
""").show(truncate=False)

StatementMeta(, 825e93ac-6f82-4525-93c8-5102639161be, 11, Finished, Available, Finished, False)

BRONZE ENCOUNTER VALIDATION
+-------------+----------+--------+
|total_records|unique_ids|null_ids|
+-------------+----------+--------+
|         2005|      2005|       0|
+-------------+----------+--------+

WATERMARK
+-------------+--------------------------+--------------------------+
|resource_type|last_successful_timestamp |updated_at                |
+-------------+--------------------------+--------------------------+
|Encounter    |2026-11-18 20:02:04.419699|2026-11-18 20:02:54.534191|
+-------------+--------------------------+--------------------------+

INGESTION LOG
+------------------------------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+--------------------------+-------------+---------------+-------------+------------------------------------------------------------------------------------